<a href="https://colab.research.google.com/github/jahnavibommana40-tech/Imtro-Build-Calculator/blob/main/Intro_Build_Calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q 'langchain>=1,<2' 'langchain-groq<2'

In [ ]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ").strip()
if not os.environ["GROQ_API_KEY"]:
    raise ValueError("Enter a Groq API key in the hidden prompt.")
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0, timeout=30, max_retries=1)
print("Model client configured. Cell 3 tests the connection.")

Enter your Groq API key: ··········
Model client configured. Cell 3 tests the connection.


In [ ]:
APPLES = "One person brings five apples and another brings ten. They put them together. How many apples are there?"
plain_reply = model.invoke([("human", APPLES)])
print(plain_reply.content)

They’re 15 apples in total – 5 + 10 = 15.


In [ ]:
role_reply = model.invoke([
    ("system", "You are a patient math tutor. Give one short sentence with the answer and its units."),
    ("human", APPLES),
])
print(role_reply.content)

There are 15 apples.


In [ ]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two whole-number quantities when they are combined."""
    return a + b

print(add.name)

add


In [ ]:
from math import isfinite, radians, tan

@tool
def tree_distance(height_m: float, angle_deg: float) -> float | str:
    """Find horizontal tree distance on level ground from height above eye level in metres and angle of elevation in degrees."""
    if not isfinite(height_m) or height_m <= 0:
        return "Invalid height. Supply a positive finite height above eye level."
    if not isfinite(angle_deg) or not 0 < angle_deg < 90:
        return "Invalid angle. Supply an angle strictly between 0 and 90 degrees."
    return round(height_m / tan(radians(angle_deg)), 2)

print(tree_distance.name)

tree_distance


In [ ]:
print("Apples:", add.invoke({"a": 5, "b": 10}))
print("Tree distance in metres:", tree_distance.invoke({"height_m": 10, "angle_deg": 30}))
print("Invalid angle:", tree_distance.invoke({"height_m": 10, "angle_deg": 0}))

Apples: 15
Tree distance in metres: 17.32
Invalid angle: Invalid angle. Supply an angle strictly between 0 and 90 degrees.


In [ ]:
from langchain.agents import create_agent

INSTRUCTIONS = (
    "You are the Intelligent Calculator. Use add for combining two whole-number "
    "quantities. Use tree_distance only for horizontal distance on level ground "
    "when height above eye level in metres and angle of elevation in degrees "
    "are supplied. For supported complete problems, call the matching tool once "
    "and use its result. If a required value is missing, ask for it before "
    "calling a tool. Never guess inputs. If a tool reports an invalid input, ask "
    "for a correction. For other requests, explain what you can calculate. "
    "Give the final answer with units."
)
agent = create_agent(model=model, tools=[add, tree_distance], system_prompt=INSTRUCTIONS)
print("Agent ready")

Agent ready


In [ ]:
from langchain_core.messages import ToolMessage

def solve(problem, *, run_name="intro_calculator"):
    return agent.invoke(
        {"messages": [("human", problem)]},
        config={"recursion_limit": 8, "run_name": run_name},
    )

def show_run(problem):
    result = solve(problem)
    call_count = 0
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            call_count += 1
            print("Tool requested:", call["name"], call["args"])
        if isinstance(message, ToolMessage):
            print("Tool returned:", message.name, message.content)
    print("Tool requests:", call_count)
    print("Final answer:", result["messages"][-1].content)
    return result

print("Viewer ready")

Viewer ready


In [ ]:
apples_run = show_run(APPLES)

Tool requested: add {'a': 5, 'b': 10}
Tool returned: add 15
Tool requests: 1
Final answer: There are **15 apples**.


In [ ]:
TREE = (
    "On level ground, the top of a tree is 10 metres above my eye level. "
    "The angle of elevation to the top is 30 degrees. "
    "How far away is the tree horizontally?"
)
tree_run = show_run(TREE)

Tool requested: tree_distance {'angle_deg': 30, 'height_m': 10}
Tool returned: tree_distance 17.32
Tool requests: 1
Final answer: The horizontal distance from you to the tree is **17.32 m**.


In [ ]:
MISSING_ANGLE = (
    "On level ground, the top of a tree is 10 metres above my eye level. "
    "How far away is the tree horizontally?"
)
missing_run = show_run(MISSING_ANGLE)

Tool requests: 0
Final answer: I can calculate the horizontal distance using the tree_distance tool, but I need the angle of elevation (in degrees). Could you please provide that?


In [ ]:
os.environ.pop("LANGSMITH_ENDPOINT", None)
print("Old endpoint cleared!")

Old endpoint cleared!


In [ ]:
import os

os.environ["LANGSMITH_AGENT_ID"] = "fleet-d75c18b09c82468bb452a254fde8c117"
os.environ["LANGSMITH_AGENT_ENVIRONMENT"] = "production"

print("Agent ID:", os.environ["LANGSMITH_AGENT_ID"])
print("Environment:", os.environ["LANGSMITH_AGENT_ENVIRONMENT"])

Agent ID: fleet-d75c18b09c82468bb452a254fde8c117
Environment: production


In [ ]:
from getpass import getpass

smith_key = getpass("Enter your LangSmith API key: ")

Enter your LangSmith API key: ··········


In [ ]:
from langsmith import Client, tracing_context

smith_client = Client(api_key=smith_key)

print("LangSmith client created!")

LangSmith client created!


In [ ]:
print("Agent ID:", os.environ.get("LANGSMITH_AGENT_ID"))
print("Environment:", os.environ.get("LANGSMITH_AGENT_ENVIRONMENT"))
print("Client ready:", smith_client is not None)

Agent ID: fleet-d75c18b09c82468bb452a254fde8c117
Environment: production
Client ready: True


In [ ]:
for name in ["solve", "APPLES", "TREE", "MISSING_ANGLE"]:
    print(name, ":", "Ready" if name in globals() else "Missing")

solve : Ready
APPLES : Ready
TREE : Ready
MISSING_ANGLE : Ready


In [ ]:
with tracing_context(enabled=True, client=smith_client):
    solve(APPLES, run_name="intro_apples")
    solve(TREE, run_name="intro_tree")
    solve(MISSING_ANGLE, run_name="intro_missing_angle")

smith_client.flush()

print("Three tracing tests completed!")

Three tracing tests completed!


In [ ]:
fresh_run = show_run(
    'One person brings seven apples and another brings eight. How many apples do they have together?'
)

Tool requested: add {'a': 7, 'b': 8}
Tool returned: add 15
Tool requests: 1
Final answer: They have a total of **15 apples**.


### Fresh Apples Test and Observations

I tested the calculator with a new apples question using 7 and 8.

The agent called the addition tool with arguments 7 and 8. The tool returned 15, and the final answer was 15 apples.

The calculator also handled the earlier apples and tree-distance questions successfully. For the tree question, it calculated approximately 17.32 metres using the given elevation angle.

When the elevation angle was missing, the agent asked the user to provide the angle instead of guessing.

In LangSmith, I observed three traces: `intro_apples`, `intro_tree`, and `intro_missing_angle`. I also tested the calculator through ChatGPT and observed successful tool responses.

The fresh test confirms that the calculator can handle different input numbers and return the correct result.

